thi script was made to run in google colab not on local desktop


modify configurations to use T4 GPU, check tha modification was successful

In [ ]:
import torch
print(torch.cuda.is_available())        # must be True
print(torch.cuda.get_device_name(0))    # should say Tesla T4

True
Tesla T4


mount drive to save progress

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/sqlcoder_ft', exist_ok=True)
OUTPUT_DIR = '/content/drive/MyDrive/sqlcoder_ft'

Mounted at /content/drive


In [ ]:
download needed depenedencies

SyntaxError: invalid syntax (1448927763.py, line 1)

In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [ ]:
restart the session than run the script to check parameters were saves

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(torch.cuda.is_available())      # True
print(torch.cuda.get_device_name(0))  # Tesla T4

Mounted at /content/drive
True
Tesla T4


In [ ]:
upload schema context to drive, import it here.

In [ ]:
import importlib.util, json

spec = importlib.util.spec_from_file_location(
    "schema_prompt",
    "/content/drive/MyDrive/Colab Notebooks/schema_prompt.py"
)
schema_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(schema_mod)
SCHEMA_CONTEXT = schema_mod.SCHEMA_CONTEXT

print(SCHEMA_CONTEXT[:200])  # sanity check — should print the start of your schema


### PostgreSQL database schema
### Schema: bl_dm (all tables are in this schema)
### The database tracks retail sales transactions with customers, employees,
### products, suppliers, store branches, 


In [ ]:
upload earlier created training pairs to drive import them here.

In [ ]:
with open('/content/drive/MyDrive/Colab Notebooks/training_pairs.json') as f:
    pairs = json.load(f)

TEMPLATE = """### Task
Generate a SQL query to answer the following question:
`{question}`

### Database Schema
{schema}

### Answer
````sql
{sql}
```"""

formatted = []
for p in pairs:
    text = TEMPLATE.format(
        question=p['question'],
        schema=SCHEMA_CONTEXT,
        sql=p['sql']
    )
    formatted.append({"text": text})

print(f"Total training examples: {len(formatted)}")

from datasets import Dataset
dataset = Dataset.from_list(formatted)
print(dataset)


Total training examples: 308
Dataset({
    features: ['text'],
    num_rows: 308
})


In [ ]:
loading the model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = "defog/sqlcoder-7b-2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

print("Model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 8,388,608 || all params: 6,746,935,296 || trainable%: 0.1243


make trainer

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/sqlcoder_ft'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(OUTPUT_DIR)

/content/drive/MyDrive/Colab Notebooks/sqlcoder_ft


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        save_steps=50,
        save_total_limit=3,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
        dataset_text_field="text",
        max_length=2048,
        packing=False,
    )
)

print("Trainer ready")

Adding EOS to train dataset:   0%|          | 0/308 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/308 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/308 [00:00<?, ? examples/s]

Trainer ready


start training

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,0.895433
20,0.644867
30,0.230588
40,0.050058
50,0.030907
60,0.024445
70,0.019448
80,0.013837
90,0.012579
100,0.013727


TrainOutput(global_step=231, training_loss=0.09082540435095628, metrics={'train_runtime': 5138.7645, 'train_samples_per_second': 0.18, 'train_steps_per_second': 0.045, 'total_flos': 7.511650873638912e+16, 'train_loss': 0.09082540435095628, 'entropy': 0.015051747439429164, 'num_tokens': 1892352.0, 'mean_token_accuracy': 0.9969467520713806, 'epoch': 3.0})

save the adapter

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

NameError: name 'trainer' is not defined

merge and export the weights

In [ ]:
import gc, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/sqlcoder_ft'
ADAPTER_DIR = f"{OUTPUT_DIR}/lora_adapter"

# Local paths - not Drive
LOCAL_MERGED = "/content/merged_model"
LOCAL_F16 = "/content/sqlcoder_ft_f16.gguf"
LOCAL_Q4 = "/content/sqlcoder_ft_q4km.gguf"
FINAL_GGUF = f"{OUTPUT_DIR}/sqlcoder_ft_q4km.gguf"

# Step 1 - merge
gc.collect()
torch.cuda.empty_cache()

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    "defog/sqlcoder-7b-2",
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True
)

print("Merging adapter...")
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

print("Saving merged model locally...")
merged_model.save_pretrained(LOCAL_MERGED, safe_serialization=True, max_shard_size="2GB")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_MERGED)
print("Merge done")

# Step 2 - clone llama.cpp and build
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements.txt
!apt-get install -y cmake
!cd /content/llama.cpp && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j4 2>&1 | tail -5

# Step 3 - convert to f16
print("Converting to f16 GGUF...")
!python /content/llama.cpp/convert_hf_to_gguf.py {LOCAL_MERGED} --outfile {LOCAL_F16} --outtype f16

# Step 4 - quantize
print("Quantizing to q4_k_m...")
!/content/llama.cpp/build/bin/llama-quantize {LOCAL_F16} {LOCAL_Q4} q4_k_m

# Step 5 - copy final GGUF to Drive
import shutil
print("Copying to Drive...")
shutil.copy(LOCAL_Q4, FINAL_GGUF)
print(f"Done: {FINAL_GGUF}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Merging adapter...
Saving merged model locally...


Writing model shards:   0%|          | 0/7 [00:00<?, ?it/s]

Merge done
fatal: destination path '/content/llama.cpp' already exists and is not an empty directory.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.13.0
-- ggml commit:  5190c2ea8
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-common
-- Configuring done (0.6s)
-- Generating done (0.6s)
-- Build files have been written to: /content/llama.cpp/build
[100%] Built target llama-server
[100%] Linking CXX executable ../bin/llama
[100%] Built target llama-app
[100%] Linking 

FileNotFoundError: [Errno 2] No such file or directory: '/content/sqlcoder_ft_q4km.gguf'

In [ ]:
LOCAL_F16 = "/content/sqlcoder_ft_f16.gguf"
LOCAL_Q4 = "/content/sqlcoder_ft_q4km.gguf"
FINAL_GGUF = f"{OUTPUT_DIR}/sqlcoder_ft_q4km.gguf"

print("Converting to f16...")
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_model --outfile {LOCAL_F16} --outtype f16

print("Quantizing...")
!/content/llama.cpp/build/bin/llama-quantize {LOCAL_F16} {LOCAL_Q4} q4_k_m

print("Copying to Drive...")
import shutil
shutil.copy(LOCAL_Q4, FINAL_GGUF)
print(f"Done: {FINAL_GGUF}")

Converting to f16...
INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00005-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00006-of-00007.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00007-of-00007.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {4096, 32016}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16